# Vector Search Setup — Built-in AI Functions

Provisions the retrieval layer for the built-in pipeline, entirely with Databricks SDK functionalities:

* **Endpoint**: a standard Vector Search endpoint (`rag-chatbot-dev`) — the serving compute that hosts the index. Standard gives ~20-50ms query latency, suited for an interactive chatbot.
* **Index**: a Delta Sync index over `rag_chatbot_dev.silver.doc_chunks_builtin`, keyed on `chunk_id`, with **managed embeddings** — Databricks computes and stores the embeddings, so no embedding code or vector columns are needed in the table.
* **Embedding model**: `databricks-gte-large-en` (1024 dimensions, 8k-token context) — the standard built-in embedding model.
* **Sync mode**: `TRIGGERED` — the index refreshes only when `sync_index()` is called (after a chunking run). No standing sync pipeline, which suits PDFs that change rarely.

## Embedding columns: `content` vs `content_to_embed`

The builtin chunk table carries two text columns:

* `content` — the raw chunk text. This is what a generation LLM should read when answering.
* `content_to_embed` — the same chunk enriched by `ai_prep_search()` with document context: title, section headers, and extracted key fields are attached to the chunk text. A bare chunk often loses context ("*It depends on the plan*" — which plan?); the enriched text restores it, so its embedding better represents the chunk's meaning and retrieval matches real queries more accurately.

The index therefore **embeds `content_to_embed`** (better retrieval) and **returns both columns** (so the chatbot can feed the clean `content` to the generation model). The pypdf table has only `content` — plain-text extraction produces no structure to enrich with.

Run this notebook after `PDF Chunking (built-in AI functions)` whenever you want the index to reflect the latest chunks.

In [0]:
# --- Configuration -------------------------------------------------------
CATALOG = "rag_chatbot_dev"
SOURCE_TABLE = f"{CATALOG}.silver.doc_chunks_builtin"   # written by PDF Chunking (built-in AI functions)
ENDPOINT_NAME = "rag-chatbot-dev"                       # dedicated Vector Search endpoint (standard)
INDEX_NAME = f"{CATALOG}.silver.doc_chunks_builtin_vs"  # the Delta Sync index (a UC entity)
PRIMARY_KEY = "chunk_id"
EMBEDDING_COLUMN = "content_to_embed"                  # context-enriched chunk text -> best retrieval
EMBEDDING_MODEL = "databricks-gte-large-en"             # built-in embedding model (1024 dims)

from databricks.sdk import WorkspaceClient
from databricks.sdk.service import vectorsearch

w = WorkspaceClient()

# --- Endpoint: create if missing (idempotent) ----------------------------
try:
    ep = w.vector_search_endpoints.get_endpoint(endpoint_name=ENDPOINT_NAME)
    print(f"Endpoint {ENDPOINT_NAME} already exists (state: {ep.endpoint_status.state})")
except Exception:
    print(f"Creating standard endpoint {ENDPOINT_NAME} (takes a few minutes to provision)...")
    w.vector_search_endpoints.create_endpoint(
        name=ENDPOINT_NAME,
        endpoint_type=vectorsearch.EndpointType.STANDARD,
    )
    print(f"Endpoint creation requested: {ENDPOINT_NAME}")

Creating standard endpoint rag-chatbot-dev (takes a few minutes to provision)...
Endpoint creation requested: rag-chatbot-dev


In [0]:
# --- Index: create if missing (idempotent) --------------------------------
existing = [idx.name for idx in w.vector_search_indexes.list_indexes(endpoint_name=ENDPOINT_NAME)]

if INDEX_NAME in existing:
    print(f"Index {INDEX_NAME} already exists on {ENDPOINT_NAME}")
else:
    print(f"Creating Delta Sync index {INDEX_NAME} on {SOURCE_TABLE}...")
    w.vector_search_indexes.create_index(
        name=INDEX_NAME,
        endpoint_name=ENDPOINT_NAME,
        primary_key=PRIMARY_KEY,
        index_type=vectorsearch.VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=vectorsearch.DeltaSyncVectorIndexSpecRequest(
            source_table=SOURCE_TABLE,
            embedding_source_columns=[
                vectorsearch.EmbeddingSourceColumn(
                    name=EMBEDDING_COLUMN,
                    embedding_model_endpoint_name=EMBEDDING_MODEL,
                )
            ],
            # Only synced columns are returned by queries — keep everything
            # the chatbot needs for retrieval and for answer generation.
            columns_to_sync=[
                "chunk_id",
                "doc_path",
                "doc_name",
                "chunk_num",
                "content",
                "content_to_embed",
                "num_chars",
                "ingested_at",
            ],
            pipeline_type=vectorsearch.PipelineType.TRIGGERED,
        ),
    )
    print(f"Index creation requested: {INDEX_NAME}")

Creating Delta Sync index rag_chatbot_dev.silver.doc_chunks_builtin_vs on rag_chatbot_dev.silver.doc_chunks_builtin...
Index creation requested: rag_chatbot_dev.silver.doc_chunks_builtin_vs


In [0]:
import time

# --- Wait for the endpoint to come ONLINE -------------------------------
deadline = time.time() + 360  # up to 6 minutes
while True:
    state = w.vector_search_endpoints.get_endpoint(endpoint_name=ENDPOINT_NAME).endpoint_status.state
    # state is an enum (e.g. EndpointStatusState.ONLINE) — compare by its value
    if "ONLINE" in str(state):
        print(f"Endpoint {ENDPOINT_NAME} is ONLINE")
        break
    if time.time() > deadline:
        raise TimeoutError(f"Endpoint still {state} after 6 minutes — re-run this cell")
    print(f"endpoint state: {state} (waiting...)")
    time.sleep(20)

# --- Wait until the index itself is ready ---------------------------------
expected_rows = spark.table(SOURCE_TABLE).count()
deadline = time.time() + 360  # up to 6 minutes
while True:
    status = w.vector_search_indexes.get_index(index_name=INDEX_NAME).status
    if status.ready:
        print(f"Index {INDEX_NAME} is ready ({status.indexed_row_count} rows indexed so far)")
        break
    if time.time() > deadline:
        raise TimeoutError("Index not ready after 6 minutes — re-run this cell")
    print(f"index provisioning: {status.message or 'waiting...'}")
    time.sleep(20)

# --- Trigger the sync (TRIGGERED mode: only happens on demand) -----------
# Safe now that the index is ready. On first creation the initial sync runs
# automatically; on later runs this picks up changes to the source table.
print(f"Triggering sync of {INDEX_NAME} ({expected_rows} rows expected)...")
w.vector_search_indexes.sync_index(index_name=INDEX_NAME)

# --- Wait until every row is indexed -------------------------------------
deadline = time.time() + 300  # up to 5 minutes
while True:
    status = w.vector_search_indexes.get_index(index_name=INDEX_NAME).status
    print(f"indexed rows: {status.indexed_row_count} / {expected_rows}")
    if status.indexed_row_count and status.indexed_row_count >= expected_rows:
        break
    if time.time() > deadline:
        raise TimeoutError("Sync not complete after 5 minutes — re-run this cell")
    time.sleep(15)

print("Sync complete — index is ready for queries.")

Endpoint rag-chatbot-dev is ONLINE
Index rag_chatbot_dev.silver.doc_chunks_builtin_vs is ready (4 rows indexed so far)
Triggering sync of rag_chatbot_dev.silver.doc_chunks_builtin_vs (4 rows expected)...
indexed rows: 4 / 4
Sync complete — index is ready for queries.


In [0]:
# --- Verification: representative retrieval queries ---------------------
test_queries = [
    "How many days of paid annual leave do full-time employees get?",
    "What should I do when a SEV1 incident is detected?",
    "What storage does the Business plan include and what does it cost?",
]

for q in test_queries:
    print(f"\n=== {q}")
    results = w.vector_search_indexes.query_index(
        index_name=INDEX_NAME,
        columns=["chunk_id", "doc_name", "chunk_num", "content"],
        query_text=q,
        num_results=3,
    )
    # Each row is the requested columns + a similarity score appended last
    for row in results.result.data_array:
        doc_name, chunk_num, content, score = row[1], row[2], row[3], row[-1]
        print(f"  {score:.4f}  {doc_name} [chunk {chunk_num}]: {content[:120].strip()}...")


=== How many days of paid annual leave do full-time employees get?
  0.6035  hr_leave_policy.pdf [chunk 0.0]: All full-time employees receive 24 days of paid annual leave per calendar year. Leave accrues
monthly at 2 days per mon...
  0.4551  incident_runbook.pdf [chunk 0.0]: SEV1 means the payment service is fully down or more than 20 percent of transactions are failing.
SEV2 means degraded p...
  0.4521  expense_policy.pdf [chunk 0.0]: Scope: This policy applies to all employees who submit business expenses.

Travel Booking: Flights must be booked at le...

=== What should I do when a SEV1 incident is detected?
  0.5835  incident_runbook.pdf [chunk 0.0]: SEV1 means the payment service is fully down or more than 20 percent of transactions are failing.
SEV2 means degraded p...
  0.4523  product_faq.pdf [chunk 0.0]: Acme Cloud Storage is a secure file synchronization service for teams. It supports files up to 50 GB
and keeps a versio...
  0.4185  expense_policy.pdf [chunk 0.0]: Scope: 